# Imports

In [1]:
import torch
import sys
import os
from torch import Tensor
sys.path.append('..')
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from definitions import EXPERIMENTS_DIR, ROOT_DIR
from src.commands.train import Trainer, Config, ArticleAsKG
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    hamming_loss,
)
from PIL import ImageColor

In [2]:
px.defaults.template = 'seaborn'

In [3]:
dir = os.path.join(ROOT_DIR, 'assets', 'articles_classification_results')

# Load the results

In [6]:
# TASK = 'hierarchical_classification'
EXPERIMENT = 'nantes-uni-2020_labs'
CORPUS = 'nantes-uni-2020'
EXPERIMENT_DIR = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
RESULTS_DIR = os.path.join(EXPERIMENT_DIR, 'results')
HISTORIES_DIR = os.path.join(EXPERIMENT_DIR, 'histories')
CONFIG_PATH = os.path.join(EXPERIMENT_DIR, 'config.json')
SPLIT_PATH = os.path.join(EXPERIMENT_DIR, 'split_sub_dsis_4.pt')

In [7]:
history = pd.concat([
    pd.read_csv(os.path.join(HISTORIES_DIR, filename))
    for filename in os.listdir(HISTORIES_DIR)
])
history = history \
    .groupby(['epoch','phase','config','trial']) \
    .mean() \
    .reset_index()
history.head()

,epoch,phase,config,trial,guidance_loss,loss,guidance_acc,guidance_f1
0,0,train,Conv_Attn_null_False,0,0.104421,0.104421,0.963618,0.405371
1,0,train,Conv_Attn_null_False,1,0.101296,0.101296,0.963431,0.400915
2,0,train,Conv_Attn_null_False,2,0.102710,0.102710,0.963150,0.401648
3,0,train,Conv_Attn_null_False,3,0.101395,0.101395,0.964366,0.409671
4,0,val,Conv_Attn_null_False,0,0.067807,0.067807,0.976649,0.450468


In [8]:
history_mean = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','config']).mean() \
    .add_suffix('_mean')

history_std = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','config']).std() \
    .add_suffix('_std')

In [9]:
history = history_mean.join(other = history_std).reset_index()
history.head()

,epoch,phase,config,guidance_loss_mean,loss_mean,guidance_acc_mean,guidance_f1_mean,guidance_loss_std,loss_std,guidance_acc_std,guidance_f1_std
0,0,train,Conv_Attn_null_False,0.102455,0.102455,0.963641,0.404401,0.001461,0.001461,0.000520,0.004018
1,0,val,Conv_Attn_null_False,0.075230,0.075230,0.973600,0.444998,0.008394,0.008394,0.002593,0.011231
2,1,train,Conv_Attn_null_False,0.048401,0.048401,0.983259,0.507806,0.002044,0.002044,0.001077,0.005598
3,1,val,Conv_Attn_null_False,0.068520,0.068520,0.976459,0.467164,0.004607,0.004607,0.001858,0.007528
4,2,train,Conv_Attn_null_False,0.036636,0.036636,0.987824,0.526725,0.000993,0.000993,0.000703,0.004939


# Load Data

In [10]:
config = Config(experiment=EXPERIMENT)
trainer = Trainer(experiment=config.experiment, config=config.training_config)
configs = trainer.get_configs()

In [11]:
data = trainer.load_data()
# data = trainer.filter_data(data)
# data = trainer.keep_largest_cc(data) if trainer.config.keep_largest_cc else data
data.split = trainer.create_or_load_split(data)
data.articles.split = data.split

In [12]:
metadata = data.metadata

# Learning Curves

In [13]:
def line_plot(
    df : pd.DataFrame,
    metric : str
) -> None:
    
    fig = go.Figure()

    colors = px.colors.qualitative.Plotly
    exprs = df['config'].unique()

    for i, expr in enumerate(exprs):

        history_val = df.query(f"phase == 'val' and config == '{expr}'")
        color = colors[i % len(colors)]

        fig.add_trace(
            go.Scatter(
                name = expr,
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean'],
                mode='lines',
                line=dict(color=color),
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']+history_val[f'{metric}_std'],
                mode='lines',
                marker=dict(color="#444"),
                line=dict(width=0),
                showlegend=False
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']-history_val[f'{metric}_std'],
                marker=dict(color="#444"),
                line=dict(width=0),
                mode='lines',
                fillcolor=f'rgb{ImageColor.getcolor(color, "RGB")}'.replace('rgb','rgba').replace(')',',0.2)'),
                fill='tonexty',
                showlegend=False
            )
        )

    fig.update_layout(
        xaxis_title='Epoch',
        yaxis_title=metric.capitalize(),
        width = 960,
        height = 800
    )

    fig.show()

In [14]:
line_plot(history, 'loss')

In [15]:
line_plot(history, 'guidance_acc')

In [16]:
line_plot(history, 'guidance_f1')

# Calculate other metrics

## Load Predictions

In [17]:
def load_results(results_dir : str) -> dict[tuple[str, str], Tensor]:

    results = dict()

    for filename in os.listdir(results_dir):

        parts = filename.split('.')[0].split('_')
        trial = parts[-1]
        expr = '_'.join(parts[1:-1])

        results[(expr,trial)] = torch.load(os.path.join(RESULTS_DIR, filename), map_location='cpu', weights_only=False)

    return results

In [18]:
results = load_results(RESULTS_DIR)

## Label Wise & Row Wise Metrics

In [19]:
def display_mean_std_df(df : pd.DataFrame) -> pd.DataFrame:

    columns = set([
        col.replace('_mean','')
        for col in df.columns if 'mean' in col
    ])

    new_df = pd.DataFrame()

    for col in df.columns:

        if not col.endswith('_mean') and not col.endswith('_std'):
            new_df[col] = df[col]

    for col in columns:
        new_df[col] = df[f'{col}_mean'].apply(lambda x : f'{x:.2%}') + ' ± ' + df[f'{col}_std'].apply(lambda x : f'{x:.2%}')

    return new_df

In [20]:
def get_experiment_params(expr : str) -> dict:

    parts = expr.split('_')
        
    return {
        "GNN Variant" : parts[0],
        "Readout Function" : parts[1],
        "Pooling Layer" : parts[2],
        "Undersampling" : parts[3]
    }

In [21]:
def get_label_wise_metrics(
    data : ArticleAsKG,
    results : dict[tuple[str,str],Tensor],
    labels : list[str]
) -> tuple[pd.DataFrame,pd.DataFrame]:

    label_wise_metrics = []

    for label in range(data.y.size(-1)):
            
        for (expr,trial),result in results.items():

            y = data.y[data.split == int(trial[-1])]
            y_hat = result[data.split == int(trial[-1])].sigmoid()
            y_hard = y_hat.gt(0.5).long()
            
            label_wise_metrics.append({
                "team" : label,
                "expr" : expr,
                "trial" : trial,
                "accuracy" : accuracy_score(y[:,label], y_hard[:,label]),
                "f1_score" : f1_score(y[:,label], y_hard[:,label]),
                "precision" : precision_score(y[:,label], y_hard[:,label]),
                "recall" : recall_score(y[:,label], y_hard[:,label]),
                "auc" : roc_auc_score(y[:,label], y_hat[:,label])
            })

    label_wise_metrics = pd.DataFrame(label_wise_metrics)
    label_wise_metrics['team'] = label_wise_metrics['team'].apply(labels.__getitem__)
    label_wise_metrics['expr'] = label_wise_metrics['expr'].apply(str)

    label_wise_metrics_agg = label_wise_metrics.drop(columns="team").groupby(["expr","trial"]).mean().reset_index().sort_values(by = 'expr')

    label_wise_metrics_mean = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    label_wise_metrics_std = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    label_wise_metrics_agg = label_wise_metrics_mean.join(label_wise_metrics_std).reset_index()

    return label_wise_metrics_agg,label_wise_metrics

In [22]:
def get_row_wise_metrics(
    data : ArticleAsKG,
    results : dict[tuple[str,str],Tensor],
) -> tuple[pd.DataFrame,pd.DataFrame]:

    row_wise_metrics = []

    for (expr,trial),result in results.items():

        y = data.y[data.split == int(trial[-1])]
        y_hat = result[data.split == int(trial[-1])].sigmoid()
        y_hard = y_hat.gt(0.5).long()

        row_wise_metrics.append({
            "expr" : expr,
            "trial" : trial,
            "accuracy" : accuracy_score(y, y_hard),
            "f1_score" : f1_score(y, y_hard , average='macro'),
            "precision" : precision_score(y, y_hard , average='macro'),
            "recall" : recall_score(y, y_hard , average='macro'),
            "auc" : roc_auc_score(y, y_hat , average='macro'),
            "hamming_loss" : hamming_loss(y, y_hard)
        })

    row_wise_metrics = pd.DataFrame(row_wise_metrics)

    row_wise_metrics_mean = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    row_wise_metrics_std = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    row_wise_metrics_agg = row_wise_metrics_mean.join(row_wise_metrics_std).reset_index()

    return row_wise_metrics_agg,row_wise_metrics

In [24]:
metadata['labels']['labs']

['gem',
 'ls2n',
 'ietr',
 'gepea',
 'lten',
 'secalim',
 'lemna',
 'imn',
 'ceisam',
 'aau',
 'iecm']

In [25]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics(data.articles, results, metadata['labels']['labs'])
display_mean_std_df(label_wise_metrics_agg)

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one class is present in y_tr

,expr,f1_score,accuracy,precision,recall,auc
0,Conv_Attn_null_False,78.54% ± 0.78%,98.11% ± 0.12%,80.55% ± 1.13%,77.06% ± 1.40%,98.46% ± 1.00%


In [26]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics(data.articles, results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning:

Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.

/home/abdelnour-main/miniconda3/envs/text2kg-main/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning:

Only one 

,expr,f1_score,hamming_loss,accuracy,precision,recall,auc
0,Conv_Attn_null_False,78.54% ± 0.78%,1.89% ± 0.12%,86.07% ± 0.78%,80.55% ± 1.13%,77.06% ± 1.40%,96.95% ± nan%


In [27]:
hparams = metrics.apply(lambda x : pd.Series(get_experiment_params(x['expr'])), axis = 1)
metrics = hparams.join(metrics, lsuffix="hparams")
metrics['Experiment'] = metrics['expr']
metrics = metrics.drop(columns = ['expr'])
metrics = metrics[[metrics.columns[-1], *metrics.columns[:-1]]]
# metrics.to_csv(os.path.join(dir, "hierarchical_teams_metrics.xlsx"), index = False)
metrics

,Experiment,GNN Variant,Readout Function,Pooling Layer,Undersampling,f1_score,hamming_loss,accuracy,precision,recall,auc
0,Conv_Attn_null_False,Conv,Attn,null,False,78.54% ± 0.78%,1.89% ± 0.12%,86.07% ± 0.78%,80.55% ± 1.13%,77.06% ± 1.40%,96.95% ± nan%


- Visualizations

In [28]:
def bar_plot(df : pd.DataFrame, metric : str) -> None:

    fig = px.bar(
        data_frame = df.sort_values(by = f'{metric}_mean'),
        x = 'expr',
        y = f'{metric}_mean',
        barmode = 'group',
        error_y = f'{metric}_std',
        text = f'{metric}_mean',
        width = 950,
        height = 800
    )

    fig.update_layout(
        xaxis_title = 'Experiment',
        yaxis_title = metric.capitalize().replace('_',' '),
    )

    fig.update_traces(
        textposition = 'outside',
        texttemplate = '%{text:.2%}'
    )

    fig.show()

In [29]:
bar_plot(label_wise_metrics_agg, 'accuracy')

In [30]:
bar_plot(label_wise_metrics_agg, 'f1_score')

In [31]:
bar_plot(row_wise_metrics_agg, 'accuracy')